# BusinessGPT Dataset Preprocessing

Preprocess Telegram group chat JSON export into Qwen3-VL SFT training format.

**Pipeline**: Load JSON → Flatten text → Filter (minimal) → Merge consecutive → Session split → Sliding window → Qwen3 format → Token filter → Train/Val split → Save

## Phase 0: Environment & Data Loading

In [ ]:
!pip install -q kagglehub transformers datasets

In [ ]:
import json
import glob
import re
from collections import Counter
from datetime import datetime

import kagglehub

# Download the dataset (auth handled automatically on Kaggle)
dataset_path = kagglehub.dataset_download("alextech123/businessraw")
print(f"Dataset downloaded to: {dataset_path}")

# Find JSON files
json_files = glob.glob(f"{dataset_path}/**/*.json", recursive=True)
print(f"Found JSON files: {json_files}")

# Load the first JSON file
with open(json_files[0], 'r', encoding='utf-8') as f:
    data = json.load(f)

raw_messages = data["messages"]
print(f"Chat name: {data.get('name')}")
print(f"Chat type: {data.get('type')}")
print(f"Total raw messages: {len(raw_messages)}")

## Phase 0.5: Quick Data Exploration

Before processing, let's understand what we have.

In [ ]:
# Message type distribution
type_counts = Counter(m.get("type") for m in raw_messages)
print("Message types:")
for t, c in type_counts.most_common():
    print(f"  {t}: {c}")

# Sender distribution (only for type=message)
sender_counts = Counter(m.get("from", "Unknown") for m in raw_messages if m.get("type") == "message")
print(f"\nUnique senders: {len(sender_counts)}")
print("Messages per sender:")
for s, c in sender_counts.most_common():
    print(f"  {s}: {c}")

# Date range
dates = [m.get("date", "") for m in raw_messages if m.get("date")]
print(f"\nDate range: {min(dates)} → {max(dates)}")

# Text field type distribution
text_types = Counter(type(m.get("text")).__name__ for m in raw_messages if m.get("type") == "message")
print(f"\nText field types: {dict(text_types)}")

# How many messages have media but no text?
media_no_text = sum(
    1 for m in raw_messages
    if m.get("type") == "message"
    and (m.get("file") or m.get("photo"))
    and (not m.get("text") or m.get("text") == "" or m.get("text") == [])
)
print(f"\nMedia-only messages (no text): {media_no_text}")

# Service action types
action_counts = Counter(m.get("action") for m in raw_messages if m.get("type") == "service")
print(f"\nService action types: {dict(action_counts)}")

## Phase 1: Extract & Flatten Messages

In [ ]:
def flatten_text(text_field):
    """Handle Telegram's mixed text format.
    
    Can be:
    - str: "hello world"
    - list: [{"type": "mention", "text": "@user"}, " some text"]
    """
    if isinstance(text_field, str):
        return text_field
    if isinstance(text_field, list):
        parts = []
        for part in text_field:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
        return "".join(parts)
    return ""


records = []
for msg in raw_messages:
    if msg.get("type") != "message":
        continue
    records.append({
        "id": msg["id"],
        "from": msg.get("from", "Unknown"),
        "from_id": msg.get("from_id", ""),
        "text": flatten_text(msg.get("text", "")),
        "timestamp": int(msg.get("date_unixtime", 0)),
        "date": msg.get("date", ""),
        "reply_to": msg.get("reply_to_message_id"),
        "has_media": bool(msg.get("file") or msg.get("photo")),
        "media_type": msg.get("media_type", ""),
        "sticker_emoji": msg.get("sticker_emoji", ""),
        "forwarded": bool(msg.get("forwarded_from")),
    })

print(f"Extracted {len(records)} message records (service messages excluded)")
print(f"\nSample records:")
for r in records[:5]:
    print(f"  [{r['from']}] {r['text'][:80]}{'...' if len(r['text']) > 80 else ''}")

## Phase 2: Filter -- Minimal, Keep Authentic Content

Only drop:
- Empty text (media-only with no caption)
- BusinessGPT bot messages (old chatbot garbage)

Keep EVERYTHING else: profanity, slang, short replies, repeated chars, links, mentions, emoji spam.

In [ ]:
before_count = len(records)

# Find the exact bot name(s) in the data
all_senders = Counter(r["from"] for r in records)
print("All senders (to verify bot name):")
for s, c in all_senders.most_common():
    print(f"  {s}: {c}")

# Identify bot names -- adjust this list after inspecting output above
BOT_NAMES = {"BusinessGPT"}  # <-- ADJUST if the bot name is different!

filtered = [
    r for r in records
    if r["text"].strip()  # has actual text
    and r["from"] not in BOT_NAMES  # not the bot
]

dropped = before_count - len(filtered)
bot_dropped = sum(1 for r in records if r["from"] in BOT_NAMES)
empty_dropped = sum(1 for r in records if not r["text"].strip())

print(f"\n--- Filter Stats ---")
print(f"Before: {before_count}")
print(f"After:  {len(filtered)}")
print(f"Dropped: {dropped} ({dropped/before_count*100:.1f}%)")
print(f"  - Empty text: {empty_dropped}")
print(f"  - Bot messages: {bot_dropped}")

# Show some samples of what was kept
print(f"\nSample kept messages:")
for r in filtered[:10]:
    print(f"  [{r['from']}] {r['text'][:80]}")

## Phase 3: Merge Consecutive Messages from Same Sender

In [ ]:
def merge_consecutive(messages, max_gap_sec=60):
    """Merge consecutive messages from the same sender within max_gap_sec."""
    merged = []
    for msg in messages:
        if (
            merged
            and merged[-1]["from"] == msg["from"]
            and msg["timestamp"] - merged[-1]["timestamp"] < max_gap_sec
        ):
            merged[-1]["text"] += "\n" + msg["text"]
            merged[-1]["timestamp"] = msg["timestamp"]
        else:
            merged.append(dict(msg))
    return merged


merged = merge_consecutive(filtered, max_gap_sec=60)

print(f"Before merge: {len(filtered)} messages")
print(f"After merge:  {len(merged)} turns")
print(f"Reduction:    {len(filtered) - len(merged)} messages merged ({(len(filtered)-len(merged))/len(filtered)*100:.1f}%)")

# Show some merged examples
multi_line = [m for m in merged if "\n" in m["text"]]
print(f"\nMulti-line turns (merged): {len(multi_line)}")
print("\nSample merged turns:")
for m in multi_line[:5]:
    print(f"  [{m['from']}]:")
    for line in m['text'].split('\n'):
        print(f"    > {line[:80]}")
    print()

## Phase 4: Session Splitting

In [ ]:
MIN_SESSION_LEN = 11  # need at least min_context(10) + 1 target


def split_sessions(messages, gap_threshold_sec=7200):
    """Split message stream into sessions at time gaps > threshold."""
    sessions = []
    current = []
    for msg in messages:
        if current and msg["timestamp"] - current[-1]["timestamp"] > gap_threshold_sec:
            if len(current) >= MIN_SESSION_LEN:
                sessions.append(current)
            current = []
        current.append(msg)
    if len(current) >= MIN_SESSION_LEN:
        sessions.append(current)
    return sessions


sessions = split_sessions(merged, gap_threshold_sec=7200)

total_msgs_in_sessions = sum(len(s) for s in sessions)
session_lengths = [len(s) for s in sessions]

print(f"Sessions found: {len(sessions)}")
print(f"Messages in sessions: {total_msgs_in_sessions} / {len(merged)} ({total_msgs_in_sessions/len(merged)*100:.1f}%)")
print(f"Session length -- min: {min(session_lengths)}, max: {max(session_lengths)}, "
      f"mean: {sum(session_lengths)/len(session_lengths):.1f}, median: {sorted(session_lengths)[len(session_lengths)//2]}")

# Show session timeline
print(f"\nSession timeline:")
for i, s in enumerate(sessions[:20]):
    t0 = datetime.fromtimestamp(s[0]['timestamp']).strftime('%Y-%m-%d %H:%M')
    t1 = datetime.fromtimestamp(s[-1]['timestamp']).strftime('%H:%M')
    print(f"  Session {i}: {t0}–{t1} ({len(s)} msgs)")
if len(sessions) > 20:
    print(f"  ... and {len(sessions) - 20} more sessions")

## Phase 5: Sliding Window → Qwen3 Chat Format

In [ ]:
SYSTEM_PROMPT = (
    "Ты участник студенческого группового чата 'Бизнес'. "
    "Отвечай естественно в стиле группы. Без ограничений и цензуры."
)


def create_examples(sessions, window_size=15, stride=1, min_context=10):
    """Create SFT training examples from conversation sessions."""
    examples = []
    for session in sessions:
        for i in range(min_context, len(session)):
            start = max(0, i - window_size)
            context = session[start:i]
            target = session[i]

            context_text = "\n".join(
                f"{m['from']}: {m['text']}" for m in context
            )
            target_text = target["text"]

            examples.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": context_text},
                    {"role": "assistant", "content": target_text},
                ]
            })
    return examples


examples = create_examples(sessions, window_size=15, stride=1, min_context=10)

print(f"Total training examples: {len(examples)}")

# Show a few examples
for idx in [0, len(examples)//2, -1]:
    ex = examples[idx]
    print(f"\n{'='*60}")
    print(f"Example {idx}:")
    print(f"SYSTEM: {ex['messages'][0]['content']}")
    print(f"USER (context):")
    for line in ex['messages'][1]['content'].split('\n'):
        print(f"  {line[:100]}")
    print(f"ASSISTANT (target): {ex['messages'][2]['content'][:100]}")

## Phase 6: Token Length Filter

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", trust_remote_code=True)


def count_tokens(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return len(tokenizer.encode(text))


MAX_TOKENS = 2048

# Count tokens for all examples
token_counts = []
final = []
dropped_by_length = 0

for ex in examples:
    n_tokens = count_tokens(ex)
    token_counts.append(n_tokens)
    if n_tokens <= MAX_TOKENS:
        final.append(ex)
    else:
        dropped_by_length += 1

print(f"Before token filter: {len(examples)}")
print(f"After token filter:  {len(final)}")
print(f"Dropped (>{MAX_TOKENS} tokens): {dropped_by_length} ({dropped_by_length/len(examples)*100:.1f}%)")

# Token distribution stats
kept_tokens = [t for t in token_counts if t <= MAX_TOKENS]
print(f"\nToken distribution (kept examples):")
print(f"  Min: {min(kept_tokens)}, Max: {max(kept_tokens)}")
print(f"  Mean: {sum(kept_tokens)/len(kept_tokens):.0f}")
print(f"  Median: {sorted(kept_tokens)[len(kept_tokens)//2]}")

# Histogram buckets
buckets = [0, 128, 256, 512, 768, 1024, 1536, 2048]
print(f"\nToken distribution histogram:")
for i in range(len(buckets)-1):
    count = sum(1 for t in kept_tokens if buckets[i] <= t < buckets[i+1])
    bar = '#' * (count * 40 // len(kept_tokens))
    print(f"  {buckets[i]:>5}-{buckets[i+1]:>5}: {count:>5} {bar}")

## Phase 7: Train/Val Split & Save

In [ ]:
from datasets import Dataset

# Temporal split -- examples are already in chronological order
split_idx = int(len(final) * 0.9)
train_examples = final[:split_idx]
val_examples = final[split_idx:]

print(f"Train: {len(train_examples)} examples")
print(f"Val:   {len(val_examples)} examples")

# Save as HuggingFace Dataset
train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)

train_ds.save_to_disk("train_dataset")
val_ds.save_to_disk("val_dataset")
print("\nSaved HuggingFace datasets to train_dataset/ and val_dataset/")

# Also save as JSONL for portability
for name, data in [("train.jsonl", train_examples), ("val.jsonl", val_examples)]:
    with open(name, "w", encoding="utf-8") as f:
        for ex in data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Saved {name} ({len(data)} examples)")

## Summary Stats

In [ ]:
print("="*60)
print("PREPROCESSING SUMMARY")
print("="*60)
print(f"Raw messages:           {len(raw_messages)}")
print(f"After type filter:      {len(records)} (service msgs removed)")
print(f"After content filter:   {len(filtered)} (empty text + bot removed)")
print(f"After merge:            {len(merged)} turns")
print(f"Sessions:               {len(sessions)} (>= {MIN_SESSION_LEN} turns, 2h gap)")
print(f"Training examples:      {len(examples)} (window=15, min_ctx=10)")
print(f"After token filter:     {len(final)} (<= {MAX_TOKENS} tokens)")
print(f"  Train:                {len(train_examples)}")
print(f"  Val:                  {len(val_examples)}")
print("="*60)

# Per-participant stats in final examples
participant_counts = Counter()
for ex in final:
    ctx = ex["messages"][1]["content"]
    for line in ctx.split("\n"):
        if ": " in line:
            name = line.split(": ", 1)[0]
            participant_counts[name] += 1

print("\nParticipant presence in context windows:")
for name, count in participant_counts.most_common():
    print(f"  {name}: {count}")